# Chapter 4 — What Happens Next?
**Which counties will lose the most forest by 2030?**

Author: Davis Mironga  
Data: Global Forest Watch — Tree cover loss by county, 2001–2024  
Method: Linear trend + Random Forest regression

---

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

DB_PATH = Path('../kenya_deforestation.db')
OUT_PATH = Path('../data/outputs')
OUT_PATH.mkdir(parents=True, exist_ok=True)

## 1. Load Data

In [ ]:
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT
        subnational_1 AS county,
        year,
        tree_cover_loss_ha AS loss_ha,
        co2_emissions_mt
    FROM gfw_tree_cover_loss
    WHERE year BETWEEN 2001 AND 2024
    ORDER BY county, year
""", conn)

conn.close()
print(f"Loaded {len(df):,} rows across {df['county'].nunique()} counties")
df.head()

## 2. Feature Engineering

In [ ]:
df = df.sort_values(['county', 'year'])

df['loss_lag1'] = df.groupby('county')['loss_ha'].shift(1)
df['loss_lag2'] = df.groupby('county')['loss_ha'].shift(2)
df['loss_roll3'] = df.groupby('county')['loss_ha'].transform(lambda x: x.rolling(3).mean())
df['cumulative_loss'] = df.groupby('county')['loss_ha'].cumsum()
df['year_index'] = df['year'] - 2001  # 0-based time index

df_clean = df.dropna()
print(f"After feature engineering: {len(df_clean):,} rows")

## 3. Baseline — Linear Trend Per County

In [ ]:
def fit_linear_trend(county_df):
    X = county_df[['year_index']]
    y = county_df['loss_ha']
    model = LinearRegression().fit(X, y)
    return model.coef_[0], model.intercept_

trends = (
    df_clean.groupby('county')
    .apply(fit_linear_trend)
    .apply(pd.Series)
    .rename(columns={0: 'slope_ha_per_year', 1: 'intercept'})
    .reset_index()
)

# Positive slope = accelerating loss
trends = trends.sort_values('slope_ha_per_year', ascending=False)
print("Counties with fastest-accelerating loss:")
trends.head(10)

## 4. Random Forest Forecast

In [ ]:
FEATURES = ['year_index', 'loss_lag1', 'loss_lag2', 'loss_roll3', 'cumulative_loss']
TARGET = 'loss_ha'

# Train on 2001–2019, validate on 2020–2024
train = df_clean[df_clean['year'] <= 2019]
val   = df_clean[df_clean['year'] >= 2020]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

val_preds = rf.predict(X_val)
mae = mean_absolute_error(y_val, val_preds)
print(f"Validation MAE: {mae:.1f} ha")

# Feature importance
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importance:")
print(importance)

## 5. Generate 2025–2030 Forecasts Per County

In [ ]:
FORECAST_YEARS = list(range(2025, 2031))
forecasts = []

for county, grp in df_clean.groupby('county'):
    grp = grp.sort_values('year').copy()

    for yr in FORECAST_YEARS:
        yi = yr - 2001
        lag1 = grp['loss_ha'].iloc[-1]
        lag2 = grp['loss_ha'].iloc[-2]
        roll3 = grp['loss_ha'].iloc[-3:].mean()
        cumul = grp['cumulative_loss'].iloc[-1]

        X_pred = pd.DataFrame([[yi, lag1, lag2, roll3, cumul]], columns=FEATURES)
        pred = max(0, rf.predict(X_pred)[0])  # loss cannot be negative

        forecasts.append({'county': county, 'year': yr, 'forecast_loss_ha': round(pred, 1)})

        # Append prediction as new row for next iteration
        new_row = grp.iloc[-1].copy()
        new_row['year'] = yr
        new_row['year_index'] = yi
        new_row['loss_ha'] = pred
        new_row['loss_lag1'] = lag1
        new_row['loss_lag2'] = lag2
        new_row['loss_roll3'] = roll3
        new_row['cumulative_loss'] = cumul + pred
        grp = pd.concat([grp, new_row.to_frame().T], ignore_index=True)

forecast_df = pd.DataFrame(forecasts)
print(f"Generated forecasts for {forecast_df['county'].nunique()} counties")
forecast_df.head(12)

## 6. Which Counties Will Lose Most by 2030?

In [ ]:
by_2030 = (
    forecast_df.groupby('county')['forecast_loss_ha']
    .sum()
    .reset_index()
    .rename(columns={'forecast_loss_ha': 'projected_loss_2025_2030_ha'})
    .sort_values('projected_loss_2025_2030_ha', ascending=False)
)

fig = px.bar(
    by_2030.head(15),
    x='projected_loss_2025_2030_ha', y='county',
    orientation='h',
    title='Projected Tree Cover Loss 2025–2030 — Top 15 Counties',
    labels={'projected_loss_2025_2030_ha': 'Projected Loss (ha)', 'county': 'County'},
    template='plotly_white',
    color='projected_loss_2025_2030_ha',
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

by_2030.to_csv(OUT_PATH / 'forecast_2030.csv', index=False)
print("Saved to data/outputs/forecast_2030.csv")

## 7. Policy Recommendations

Based on forecast results — fill in after running analysis:

| Priority | County | Projected Loss (ha) | Recommended Intervention |
|----------|--------|---------------------|---------------------------|
| 1 | *TBD* | *TBD* | Strengthen community forest associations (CFAs) |
| 2 | *TBD* | *TBD* | Regularize land tenure for at-risk communities |
| 3 | *TBD* | *TBD* | Expand agroforestry incentives |

Communities at highest risk (cross-reference with Chapter 3 poverty data):
- **Ogiek** — Mau Forest complex
- **Sengwer** — Cherangany Hills
- **Pastoralists** — Baringo, Tana River

---
**Data needed:** `data/raw/gfw_tree_cover_loss.csv` loaded into the database via `sql/02_load_data.sql`.